# bf

In [10]:
import os
import glob
import pandas as pd
import numpy as np
import zipfile
from scipy.fft import fft, fftfreq
from scipy.signal import medfilt
import matplotlib.pyplot as plt

# ============================================================
# 設定與路徑
# ============================================================
base_path = r"E:\EarthScienceFair_Data"
target_folders = ["5"]
WINDOW_SIZE = 150 
STRIDE = 5       

# ============================================================
# 工具函式
# ============================================================

def clean_for_window(raw_data):
    """將原始欄位轉數值、插值補坑、去中心化"""
    s = pd.to_numeric(pd.Series(raw_data), errors='coerce')
    if s.isna().all(): return np.array([])
    s = s.interpolate(method='linear', limit_direction='both').ffill().bfill()
    val = s.values
    return val - np.mean(val) if len(val) > 0 else np.array([])

def get_sliding_freq(time, signal, win_size, stride):
    t_res, f_res = [], []
    for start in range(0, len(signal) - win_size, stride):
        end = start + win_size
        seg_t = time[start:end]
        seg_s = signal[start:end]
        
        N = len(seg_s)
        dt = np.mean(np.diff(seg_t))
        if dt <= 0: continue
        
        yf = fft(seg_s)
        xf = fftfreq(N, dt)[:N//2]
        amp = 2.0/N * np.abs(yf[:N//2])
        
        # 刪除頻率範圍過濾，直接取最大振幅對應頻率
        mask = (xf >= 1.0) & (xf <= 4.0)
        if np.any(mask) and np.sum(amp[mask]) > 0:
            f_main = np.sum(xf[mask] * amp[mask]) / np.sum(amp[mask])
            t_res.append(np.mean(seg_t))
            f_res.append(f_main)
    return np.array(t_res), np.array(f_res)

# ============================================================
# 主流程
# ============================================================
xlsx_files = []
for folder in target_folders:
    fp = os.path.join(base_path, folder)
    if os.path.exists(fp):
        found = glob.glob(os.path.join(fp, "**", "*.xlsx"), recursive=True)
        print(f"資料夾 {fp} 找到 {len(found)} 個 xlsx 檔案")
        xlsx_files.extend(found)
    else:
        print(f"資料夾不存在: {fp}")

for tracker_file in xlsx_files:
    file_id = os.path.splitext(os.path.basename(tracker_file))[0]
    print(f"\n處理中: {file_id}")
    
    try:
        df = pd.read_excel(tracker_file, header=2) 
        print(f"  xlsx 讀取成功，shape={df.shape}")

        # 1. 讀取時間軸 (強制歸零)
        df_clean = df[pd.to_numeric(df['t_s'], errors='coerce').notna()].copy()
        df_clean = df_clean.reset_index(drop=True)
        print(f"  有效行數: {len(df_clean)}/{len(df)} 筆")

        t_s_raw = pd.to_numeric(df_clean['t_s'], errors='coerce').values
        t_s = t_s_raw - t_s_raw[0]
        
        # 2. 讀取目標層位，修復 yi 的保護
        target_layers = {
            "ya": df_clean.iloc[:, 1].values if df_clean.shape[1] > 1 else None,
            "yd": df_clean.iloc[:, 4].values if df_clean.shape[1] > 4 else None,
            "yh": df_clean.iloc[:, 8].values if df_clean.shape[1] > 8 else None,
            "yi": df_clean.iloc[:, -9].values if df_clean.shape[1] >= 9 else None
        }

        plt.figure(figsize=(12, 6))

        # ↓ csv 區段整個移除，直接畫 xlsx 層位
        colors = {"ya":"red", "yd":"orange", "yh":"green", "yi":"blue"}
        for name, raw_val in target_layers.items():
            if raw_val is None: continue
            
            clean_sig = clean_for_window(raw_val)
            print(f"  層位 {name}: clean 後長度={len(clean_sig)}")
            if len(clean_sig) > WINDOW_SIZE:
                sig_med = medfilt(clean_sig, kernel_size=5)
                tp, fp = get_sliding_freq(t_s, sig_med, WINDOW_SIZE, STRIDE)
                print(f"  層位 {name}: 頻率範圍 = {fp.min():.3f} ~ {fp.max():.3f} Hz")
                plt.plot(tp, fp, color=colors[name], label=f"Layer {name}", alpha=0.7)

        plt.title(f"Sliding Window Analysis (xlsx only): {file_id}")
        plt.xlabel("Time (s)")
        plt.ylabel("Frequency (Hz)")
        plt.grid(True, alpha=0.3)
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig(f"Sliding_{file_id}.png", dpi=150)
        plt.close()

        print(f"  t_s 範圍: {t_s[0]:.4f} ~ {t_s[-1]:.4f} s")
        print(f"  dt 平均: {np.mean(np.diff(t_s)):.6f}, 最小: {np.min(np.diff(t_s)):.6f}, 負值數量: {np.sum(np.diff(t_s) <= 0)}")

        # 測試第一個窗口
        seg_s = medfilt(clean_for_window(df.iloc[:, 1].values), kernel_size=5)[:WINDOW_SIZE]
        seg_t = t_s[:WINDOW_SIZE]
        dt_test = np.mean(np.diff(seg_t))
        print(f"  第一窗口 dt={dt_test:.6f}")
        yf = fft(seg_s)
        xf = fftfreq(WINDOW_SIZE, dt_test)[:WINDOW_SIZE//2]
        amp = 2.0/WINDOW_SIZE * np.abs(yf[:WINDOW_SIZE//2])
        print(f"  xf 範圍: {xf.min():.3f} ~ {xf.max():.3f} Hz")
        print(f"  amp 最大值: {amp.max():.6f}, argmax={np.argmax(amp)}")
        print(f"  對應頻率: {xf[np.argmax(amp)]:.4f} Hz")

        # 印出 1~4 Hz 之間的所有頻率和振幅
        mask = (xf >= 1.0) & (xf <= 4.0)
        print("  1~4Hz 頻譜：")
        for f, a in zip(xf[mask], amp[mask]):
            print(f"    {f:.3f} Hz: {a:.6f}")

    except Exception as e:
        print(f"失敗 {file_id}: {e}")

資料夾 E:\EarthScienceFair_Data\5 找到 22 個 xlsx 檔案

處理中: 5-G1-1
  xlsx 讀取成功，shape=(1695, 18)
  有效行數: 470/1695 筆
  層位 ya: clean 後長度=470
  層位 ya: 頻率範圍 = 1.753 ~ 2.330 Hz
  層位 yd: clean 後長度=470
  層位 yd: 頻率範圍 = 1.647 ~ 2.334 Hz
  層位 yh: clean 後長度=0
  層位 yi: clean 後長度=470
  層位 yi: 頻率範圍 = 1.618 ~ 2.347 Hz
  t_s 範圍: 0.0000 ~ 15.6350 s
  dt 平均: 0.033337, 最小: 0.033330, 負值數量: 0
  第一窗口 dt=0.033333
  xf 範圍: 0.000 ~ 14.800 Hz
  amp 最大值: 0.024290, argmax=0
  對應頻率: 0.0000 Hz
  1~4Hz 頻譜：
    1.200 Hz: 0.005579
    1.400 Hz: 0.008464
    1.600 Hz: 0.007651
    1.800 Hz: 0.005798
    2.000 Hz: 0.003240
    2.200 Hz: 0.002462
    2.400 Hz: 0.001465
    2.600 Hz: 0.000814
    2.800 Hz: 0.001252
    3.000 Hz: 0.000868
    3.200 Hz: 0.000325
    3.400 Hz: 0.000413
    3.600 Hz: 0.000555
    3.800 Hz: 0.000525
    4.000 Hz: 0.000580

處理中: 5-G1-2
  xlsx 讀取成功，shape=(1934, 18)
  有效行數: 524/1934 筆
  層位 ya: clean 後長度=524
  層位 ya: 頻率範圍 = 1.721 ~ 2.537 Hz
  層位 yd: clean 後長度=524
  層位 yd: 頻率範圍 = 1.629 ~ 2.463 Hz
  層位 yh: 